In [9]:
import pandas as pd
import numpy as np
import vowpalwabbit 
import re
from vowpalwabbit import pyvw
import random

In [10]:
def load_data(path):
    df = pd.read_csv(path)
    return df

def filter_top_products(df, n):
    top = (df.groupby('ASIN')['reviewerID'].count().nlargest(n).index)
    filtered = df[df['ASIN'].isin(top)].copy()
    print(len(filtered))
    return filtered

def add_weekend(df):
    df['reviewDate'] = pd.to_datetime(df['reviewDate'])
    df['is_weekend'] = df['reviewDate'].dt.dayofweek.isin([5,6]).astype(int)
    return df

def add_synthetic_context(df, seed=42):
    rng = np.random.default_rng(seed)
    n=len(df)

    df['time_of_day'] = rng.choice(
        ['morning','afternoon','evening','night'],
        size=n, p=[0.25, 0.30, 0.30, 0.15]
    )
    df['device']  = rng.choice(
        ['mobile','desktop','tablet'],
        size=n, p=[0.55, 0.35, 0.10]
    )
    df['intent'] = rng.choice(
        ['browsing','buying'],
        size=n, p=[0.60, 0.40]
    )
    return df

def add_reward(df):
    product_avg = df.groupby('ASIN')['customerReview'].transform('mean')
    product_median = df.groupby('ASIN')['customerReview'].transform('median')
    df['reward'] = (df['customerReview']== 5).astype(int)
    print(f"Reward rate: {df['reward'].mean()}")
    return df

def build_product_features(df):
    products = (df.groupby('ASIN').agg(
        ProductName=('ProductName', 'first'),
        category=('category', 'first'),
        avg_rating=('customerReview', 'mean'),
        review_count=('customerReview', 'count'),
    ).reset_index())

    products['avg_rating'] = products['avg_rating'].round(2)
    products['log_review_count'] = np.log1p(products['review_count']).round(3)

    products['name_words'] = (
        products['ProductName'].str.lower().
        str.replace('_',' ', regex=False).str.replace(r'[^a-z0-9]',' ', regex=True).str.strip()
        )
    print(f"Products: {len(products)}")
    return products

def build_interactions(df,products, seed=42):
    interactions = df[['reviewerID', 'ASIN','reviewLocation','is_weekend','time_of_day','device','intent','reward']].merge(
        products[['ASIN','category','avg_rating','log_review_count','name_words']], on='ASIN',how='left'
    )

    interactions = interactions.sample(frac=1, random_state=seed).reset_index(drop=True)
    return interactions

def save(products, interactions):
    products.to_csv('output/products.csv', index=False)
    interactions.to_csv('output/interactions.csv', index=False)


file_path = r'C:\Users\dcheruiyot2\OneDrive - KPMG\Projects\rec-sys\dataset\utility\reviews.csv'
df=load_data(file_path)
df=filter_top_products(df,200)
df=add_weekend(df)
df=add_synthetic_context(df)
df=add_reward(df)
products = build_product_features(df)
interactions = build_interactions(df, products)
save(products, interactions)





5524
Reward rate: 0.7666545981173063
Products: 200


In [11]:
def _clean(text):
    return re.sub(r"[|:\s]+","_", str(text).lower())

def build_shared_context(context):
    location = _clean(context.get('location','unknown'))
    time_of_day = _clean(context.get('time_of_day','unknown'))
    device = _clean(context.get('device','unknown'))
    intent = _clean(context.get('intent','unknown'))
    is_weekend = _clean(context.get('is_weekend','unknown'))
    return (
        f"shared |user "
        f"location_{location}=1 "
        f"time_{time_of_day}=1 "
        f"device_{device}=1 "
        f"intent_{intent}=1 "
        f"is_weekend={is_weekend}"
    )

def build_arm_line(arm_index, product, cost=None, prob=None):
    category = _clean(product.get('category','unknown'))
    rating = round(float(product.get('avg_rating', 3.0)), 2)
    log_count = round(float(product.get('log_review_count', 1.0)), 3)
    words = product.get('name_words','')
    label = f"{arm_index}:{cost}:{prob}" if cost is not None else str(arm_index)

    return (
        f"{label} |item "
        f"cat_{category}=1 "
        f"rating_{rating} "
        f"log_count={log_count} "
        f"{words}"
    )

def build_vw_example(context, products, chosen_index=None, cost=None, prob=None):
    lines = [build_shared_context(context)]
    for i, product in enumerate(products):
        if chosen_index is not None and i == chosen_index:
            lines.append(build_arm_line(i,product=product, cost=cost, prob=prob))
        else:
            lines.append(build_arm_line(i, product))
    return '\n'.join(lines)

def sample_pmf(scores):    
    r=random.random()
    cummulative=0.0
    for index, prob in enumerate(scores):
        cummulative += prob
        if r <= cummulative:
            return index, prob
    return len(scores)-1, scores[-1]


In [12]:

sample_context = {
    'location':'United States',
    'time_of_day': 'morning',
    'device':'mobile',
    'intent':'buying',
    'is_weekend':1,
}

sample_products = (products.groupby('category').first().reset_index().head(3).to_dict("records"))
for i, p in enumerate(sample_products):
    line= build_arm_line(i,p)
    print(f"Arm {i}: {line}")
    # print(f"Arm {i}")
    # print(f"Category: {p['category']}")
    # print(f"Rating: {p['avg_rating']}")
    # print(f"Name: {p['name_words']}")
# build_vw_example(sample_context, sample_products)

Arm 0: 0 |item cat_bathroom=1 rating_4.0 log_count=3.296 lnx womens linen pants high waisted wide leg drawstring casual loose trousers with pockets
Arm 1: 1 |item cat_bedroom=1 rating_4.92 log_count=3.296 lutron caseta wireless pedestal for pico smart remote lped1wh white
Arm 2: 2 |item cat_books=1 rating_4.91 log_count=3.526 the song of achilles  a novel


In [13]:
# exampletest
#  arm 0  chosen and got cost=-1
training_example = build_vw_example(sample_context, sample_products, chosen_index=0, cost=-1.0,prob=1/3)

print(f"training example: {training_example}")

random.seed(42)
model = pyvw.Workspace("--cb_explore_adf --epsilon 0.2 -q:: --quiet")
pred_ex = build_vw_example(sample_context, sample_products)
for step in range(200):
    scores = model.predict(pred_ex)
    chosen_index, prob= sample_pmf(scores)
    reward=1.0 if chosen_index==0 else 0.0
    cost = -reward

    learn_ex = build_vw_example(sample_context,sample_products, chosen_index=chosen_index, cost=cost,prob=prob)
    model.learn(learn_ex)

    if step%10 ==0:
        s=model.predict(pred_ex)
        print([round(x,3) for x in s])




training example: shared |user location_united_states=1 time_morning=1 device_mobile=1 intent_buying=1 is_weekend=1
0:-1.0:0.3333333333333333 |item cat_bathroom=1 rating_4.0 log_count=3.296 lnx womens linen pants high waisted wide leg drawstring casual loose trousers with pockets
1 |item cat_bedroom=1 rating_4.92 log_count=3.296 lutron caseta wireless pedestal for pico smart remote lped1wh white
2 |item cat_books=1 rating_4.91 log_count=3.526 the song of achilles  a novel
[0.333, 0.333, 0.333]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]
[0.867, 0.067, 0.067]


In [14]:
product_lookup = products.set_index('ASIN').to_dict('index')
all_asins = list(product_lookup.keys())

total_reward = 0
total_interactions = 0

for _, row in interactions.head(500).iterrows():
    chosen_asin = row['ASIN']
    if chosen_asin not in product_lookup:
        continue

    context = {
        'location': row['reviewLocation'],
        'time_of_day': row['time_of_day'],
        'device': row['device'],
        'intent': row['intent'],
        'is_weekend': row['is_weekend'],
    }

    others = [a for a in all_asins if a!= chosen_asin]
    sampled = np.random.choice(others, size=4, replace=False).tolist()
    arm_products = [product_lookup[a] for a in [chosen_asin]+sampled]

    prob=1.0/len(arm_products)
    reward=float(row['reward'])
    cost=-reward

    model.learn(build_vw_example(context=context, products=arm_products, chosen_index=0, cost=cost, prob=prob ))
    total_reward += reward
    total_interactions += 1

print(f"Interactions: {total_interactions}")
print(f"reward: {total_reward}")
print(f"rr: {total_reward/total_interactions}")

Interactions: 500
reward: 391.0
rr: 0.782


In [15]:
# predict top 5 -morning weekday
context_morning={
    'location':'United States',
    'time_of_day': 'morning',
    'device':'mobile',
    'intent':'buying',
    'is_weekend':0,
}

all_products = products.to_dict('records')
scores = model.predict(build_vw_example(context_morning, all_products))
results = products.copy()
results['score'] = scores
top5=results.nlargest(5, 'score')
top5[['name_words','category','avg_rating','score']].reset_index(drop=True)

,name_words,category,avg_rating,score
0,broom and dustpan broom and dustpan set for ho...,Cleaning Material,4.54,0.801
1,the song of achilles a novel,Books,4.91,0.001
2,the good daughter a novel,Books,4.69,0.001
3,wrong place wrong time a reese s book club pick,Books,4.38,0.001
4,last seen alone,Books,4.54,0.001


In [16]:
# evening weekend
context_evening={
    'location':'United States',
    'time_of_day': 'evening',
    'device':'mobile',
    'intent':'buying',
    'is_weekend':1,
}

scores_eve = model.predict(build_vw_example(context_evening, all_products))
results_eve = products.copy()
results_eve['score'] = scores_eve
top5_eve = results_eve.nlargest(5, 'score')
top5_eve[['name_words','category','avg_rating','score']]

,name_words,category,avg_rating,score
156,broom and dustpan broom and dustpan set for ho...,Cleaning Material,4.54,0.801
0,the song of achilles a novel,Books,4.91,0.001
1,the good daughter a novel,Books,4.69,0.001
2,wrong place wrong time a reese s book club pick,Books,4.38,0.001
3,last seen alone,Books,4.54,0.001


In [26]:
# products[['ASIN','category', 'avg_rating','log_review_count','name_words']].head(3).to_dict(orient='records')
products[['ASIN','category', 'avg_rating','log_review_count','name_words']].groupby('category').first().reset_index().head(3).to_dict("records")

[{'category': 'Bathroom',
  'ASIN': 'b082klxtbn',
  'avg_rating': 4.0,
  'log_review_count': 3.296,
  'name_words': 'lnx womens linen pants high waisted wide leg drawstring casual loose trousers with pockets'},
 {'category': 'Bedroom',
  'ASIN': 'b003wgu2mu',
  'avg_rating': 4.92,
  'log_review_count': 3.296,
  'name_words': 'lutron caseta wireless pedestal for pico smart remote lped1wh white'},
 {'category': 'Books',
  'ASIN': '0062060627',
  'avg_rating': 4.91,
  'log_review_count': 3.526,
  'name_words': 'the song of achilles  a novel'}]

In [27]:
products.head(1).to_dict()

{'ASIN': {0: '0062060627'},
 'ProductName': {0: 'the_song_of_achilles:_a_novel'},
 'category': {0: 'Books'},
 'avg_rating': {0: 4.91},
 'review_count': {0: 33},
 'log_review_count': {0: 3.526},
 'name_words': {0: 'the song of achilles  a novel'}}

In [1]:
from bandit import ContextualBandit, build_vw_example
import pandas as pd

products = pd.read_csv('output/products.csv')
interactions = pd.read_csv('output/interactions.csv')

bandit = ContextualBandit(0.5)
all_products=products.to_dict('records')
bandit.offline_train(interactions, products, top_k=5)

ctx1={'location':'US','time_of_day':'morning', 'device':'mobile', 'intent':'browsing','is_weekend':0}
ctx2={'location':'US','time_of_day':'evening', 'device':'desktop', 'intent':'buying','is_weekend':1}

# _,_, scores1 = bandit.predict(ctx1, all_products)
# _,_, scores2 = bandit.predict(ctx2, all_products)

results = products.copy()
# results['morning'] = scores1
# results['evening'] = scores2
# results['diff'] = (results['morning'] - results['evening']).abs()


# results.nlargest(10,'diff')[['name_words','category','morning','evening']]
morningex=build_vw_example(ctx1, all_products)
eveningex=build_vw_example(ctx2, all_products)

raw_morning=bandit.model.predict(morningex)
raw_evening=bandit.model.predict(eveningex)

results['raw_morn'] = raw_morning
results['raw_even'] = raw_evening
results['raw_diff'] = (results['raw_morn'] - results['raw_even']).abs()
results['raw_diff'].max()

np.float64(4.656612873077393e-10)